In [1]:
#imports

import pandas as pd
import numpy as np
import random
from collections import Counter
from ingredient_normalisation import normalise_user_ingredients
from evaluation_queries import make_random_queries, make_manual_queries, MIN_SOURCE_INGREDIENTS
from baseline_retrieval import tfidf_model, jaccard_model, minilm_model, recipes_dataset_normalised

# random seeds
split_seed = 34
validation_seed = 83

# evaluation settings
# retrieve top 100 candidates before reranking
candidate_k = 100
# compare recommendations at top 5 and top 10
evaluation_k = [5, 10]
# query settings
ingredients_per_query = 3
query_lengths = [3, 4, 5, 6]
validation_queries = 50

# check number of recipes and columns loaded
print("dataset shape:", recipes_dataset_normalised.shape)

c:\Users\preen\Documents\AIML339 - FINAL\AIML339 - FINAL\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


dataset shape: (10000, 6)


In [2]:
# CREATE VALIDATION AND TEST SET

# select recipes with at least 5 different ingredients
valid_recipes = recipes_dataset_normalised[recipes_dataset_normalised["processing_ingredients"].apply(lambda ingredients: len(set(ingredients)) >= 5)]

# shuffle the source recipe indices using chosen seed 37
# shuffle the source recipe indices using chosen seed 34
recipe_indices = valid_recipes.index.tolist()
random.Random(split_seed).shuffle(recipe_indices)

# split  recipes into validation and test groups
split_point = int(len(recipe_indices) * 0.5)
validation_indices = recipe_indices[:split_point]
test_indices = recipe_indices[split_point:]



In [3]:
# GENERATE QUERIES 

#generate queiries from source, randomly select recipe with no repeats, set chosen seed and create empty list to store queries
def generate_queries(recipe_indices, number_of_queries, seed):
    # initialise random generator using chosen seed
    rng = random.Random(seed)
    # sample source recipes without repeats
    selected_indices = rng.sample(recipe_indices, number_of_queries)
    queries = []

    # go through each selected source recipe - remove duplicate ingredients and sort
    #randomly select ingredients for query, store source recipe and ingredients in queries
    for recipe_index in selected_indices:
        ingredients = sorted(set(recipes_dataset_normalised.loc[recipe_index, "processing_ingredients"]))
        # select requested number of ingredients from source recipe
        selected_ingredients = rng.sample(ingredients, ingredients_per_query)
        # store source recipe index and ingredient query
        queries.append({"source_recipe": recipe_index, "query": ", ".join(selected_ingredients)})

    # return as dataframe
    return pd.DataFrame(queries)


In [4]:
# GENERATE VALIDATION AND MANUAL QUERIES

# generate the same 50 validation source recipes with 3, 4, 5 and 6 ingredients
# use fixed seeds 34 and 83 - queries are generated again each time
validation_query_set = make_random_queries(recipes_dataset_normalised, partition="validation", query_lengths=query_lengths, number_of_queries=validation_queries, split_seed=split_seed, query_seed=validation_seed, min_source_ingredients=MIN_SOURCE_INGREDIENTS)

# create the 20 fixed manual queries - 5 groups with 3, 4, 5 and 6 ingredients
manual_query_set = make_manual_queries()

# check number of generated validation queries
print("validation queries:", len(validation_query_set))
# display source recipes and generated ingredient queries
display(validation_query_set)


validation queries: 200


,query_id,partition,query_number,source_index,query_length,query,split_seed,query_seed
0,validation_001_3,validation,1,1409,3,"onion, salt, barbecue sauce",34,83
1,validation_001_4,validation,1,1409,4,"onion, salt, barbecue sauce, chili powder",34,83
2,validation_001_5,validation,1,1409,5,"onion, salt, barbecue sauce, chili powder, kid...",34,83
3,validation_001_6,validation,1,1409,6,"onion, salt, barbecue sauce, chili powder, kid...",34,83
4,validation_002_3,validation,2,7476,3,"vegetable oil, soy sauce, fresh mushroom",34,83
...,...,...,...,...,...,...,...,...
195,validation_049_6,validation,49,179,6,"butter, water, cheddar cheese, egg, milk, grits",34,83
196,validation_050_3,validation,50,4910,3,"egg, sugar, flour",34,83
197,validation_050_4,validation,50,4910,4,"egg, sugar, flour, salt",34,83
198,validation_050_5,validation,50,4910,5,"egg, sugar, flour, salt, vanilla",34,83


In [5]:
# EVALUATION - RELEVANCE

#assign relevance score based on matched & missing ingredients
def recipe_relevance(user_ingredients, recipe_ingredients):

    # convert ingredients from user and recipe to sets
    # convert ingredients from user and recipe to sets
    user_set = set(user_ingredients)
    recipe_set = set(recipe_ingredients)

    # calculate matched and missing ingredients
    matched = len(user_set & recipe_set)
    missing = len(recipe_set - user_set)

    # no ingredients matched
    if matched == 0:
        return 0

    # some ingredients matched
    if matched < len(user_set):
        return 1

    # all ingredients matched, with few missing
    if missing <= 2:
        return 3

    # all ingredients matched, but more are missing
    return 2

In [6]:
# CCALCULATE RELEVANCE

# relecance for each recipe using user query
def prepare_query_relevance(user_input):
    #normalise ingredients and convert to set
    user_ingredients = normalise_user_ingredients(user_input)
    # convert ingredients from user and recipe to sets
    user_set = set(user_ingredients)

    # relevance score assigned to each recipe - mark recipes containing all ingredients as relevant
    relevance_scores = recipes_dataset_normalised["processing_ingredients"].apply(lambda ingredients: recipe_relevance(user_ingredients, ingredients))
    binary_relevance = recipes_dataset_normalised["processing_ingredients"].apply(lambda ingredients: user_set.issubset(set(ingredients)))
    # total no. of relevant recipes
    total_relevant = int(binary_relevance.sum())

    return relevance_scores, binary_relevance, total_relevant


In [7]:
#PRECISION, RECALL, NDGC, COVERAGE + MISSING INGREDIENTS

def evaluate_recommendations(recommendations, user_input, k, relevance_scores, binary_relevance, total_relevant):
    # select top k
    top_recipes = recommendations.head(k)
    user_set = set(normalise_user_ingredients(user_input))
    #calculate found relevant in topk
    relevant_found = int(binary_relevance.loc[top_recipes.index].sum())
    # proportion of top k recipes containing all user ingredients
    precision = relevant_found / k
    # proportion of all relevant recipes retrieved - undefined if none are relevant
    recall = relevant_found / total_relevant if total_relevant > 0 else np.nan

    # calculate scores in recommendation orders
    grades = relevance_scores.loc[top_recipes.index].to_numpy()
    #so lower ranked recipes contribute less
    discounts = np.log2(np.arange(2, len(grades) + 2))
    # using relevance scores and rank discounts
    dcg = np.sum((2 ** grades - 1) / discounts)

    #
    # select highest relevance scores in dataset for ideal ranking
    ideal_grades = np.sort(relevance_scores.to_numpy())[::-1][:k]
    ideal_discounts = np.log2(np.arange(2, len(ideal_grades) + 2))
    ideal_dcg = np.sum((2 ** ideal_grades - 1) / ideal_discounts)
    # compare recommendation ranking with ideal ranking
    ndcg = dcg / ideal_dcg if ideal_dcg > 0 else np.nan

    # empty list to add
    coverages = []
    missing_counts = []

    # calculate coverage and missing ingredients for each recommended recipe
    for ingredients in top_recipes["processing_ingredients"]:
        recipe_set = set(ingredients)
        # proportion of user ingredients found in recipe
        coverages.append(len(user_set & recipe_set) / len(user_set))
        # count recipe ingredients user doesnt have
        missing_counts.append(len(recipe_set - user_set))

    return {"precision": precision, "recall": recall, "ndcg": ndcg, "coverage": np.mean(coverages), "missing": np.mean(missing_counts)}


In [8]:
# COMBINED RERANKING V1 AND V2

# calculate how frequently each ingredient occurs in the recipe dataset

ingredient_counts = Counter()

# go through ingredients of each recipe
for ingredients in recipes_dataset_normalised["processing_ingredients"]:
    # count each ingredient once per recipe
    ingredient_counts.update(set(ingredients))

# proportion of recipes containing each ingredient
ingredient_frequencies = {ingredient: count / len(recipes_dataset_normalised) for ingredient, count in ingredient_counts.items()}

# calculate features for each candidate recipe
def prepare_reranking_features(candidates, user_input, similarity_column):
    # candidate recipes copy to store new reranking scores
    reranked = candidates.copy()
    user_set = set(normalise_user_ingredients(user_input))
    # scale similarity scores between 0 and 1
    scores = reranked[similarity_column]
    #lowest and highest
    minimum = scores.min()
    maximum = scores.max()

    # min-max scaling - use 0 if all candidate similarities are equal
    reranked["scaled_similarity"] = 0.0 if maximum == minimum else (scores - minimum) / (maximum - minimum)
    #proportion of user query in recipe
    reranked["coverage"] = reranked["processing_ingredients"].apply(lambda ingredients: len(user_set & set(ingredients)) / len(user_set))
    # proportion of recipe ingredients user doesnt have - original v1 penalty
    reranked["missing_penalty_v1"] = reranked["processing_ingredients"].apply(lambda ingredients: len(set(ingredients) - user_set) / len(set(ingredients)) if ingredients else 0)
    #ingredients user doesnt have
    reranked["missing_count"] = reranked["processing_ingredients"].apply(lambda ingredients: len(set(ingredients) - user_set))
    # reward common missing ingredients
    reranked["common_reward"] = reranked["processing_ingredients"].apply(lambda ingredients: sum(ingredient_frequencies.get(i, 0) for i in set(ingredients) - user_set) / len(set(ingredients)) if ingredients else 0)

    # return candidate recipes with calculated reranking features
    return reranked

# rerank
def rerank_configuration(features, coverage_weight=0.5, missing_weight=0.4, common_weight=0.1, missing_scale=5, missing_method="v2"):
    # copy features so each configuration keeps its own scores
    reranked = features.copy()
    # use original v1 proportion or v2 missing count - cap v2 penalty at 1
    missing_penalty = reranked["missing_penalty_v1"] if missing_method == "v1" else (reranked["missing_count"] / missing_scale).clip(upper=1)
    # calculate final score
    reranked["final_score"] = reranked["scaled_similarity"] + coverage_weight * reranked["coverage"] - missing_weight * missing_penalty + common_weight * reranked["common_reward"]
    # return in order
    # return candidate recipes with calculated reranking features
    return reranked.sort_values("final_score", ascending=False)

In [9]:
# V3 - COVERAGE FIRST RERANKING

# function to rank by coverage first, then similarity with missing ingredient penalty
def rerank_v3(features, missing_weight=0.4, smoothing=3):
    # candidate recipes copy to store new reranking scores
    # copy features so each configuration keeps its own scores
    reranked = features.copy()
    # ingredients missing in recipe
    # smooth missing count into penalty below 1 - larger smoothing reduces penalty
    reranked["missing_penalty_v3"] = reranked["missing_count"] / (reranked["missing_count"] + smoothing)
    # calculate final score
    # subtract weighted missing ingredient penalty from scaled similarity
    reranked["final_score"] = reranked["scaled_similarity"] - missing_weight * reranked["missing_penalty_v3"]
    # return in order
    # rank highest coverage first, then highest final score for equal coverage
    return reranked.sort_values(["coverage", "final_score"], ascending=[False, False])

In [10]:
# CONFIGURATIONS TO COMPARE

# original v1 and v2 weights
# original v1 and v2 weights, test each feature
configurations = {
    "Baseline": ("baseline", {}),
    "Combined V1": ("v1v2", {"coverage_weight": 0.3, "missing_weight": 0.2, "common_weight": 0.1, "missing_method": "v1"}),
    "Combined V2": ("v1v2", {"coverage_weight": 0.5, "missing_weight": 0.4, "common_weight": 0.1}),
    "Coverage only": ("v1v2", {"coverage_weight": 0.5, "missing_weight": 0.0, "common_weight": 0.0}),
    "V3 missing 0.0": ("v3", {"missing_weight": 0.0, "smoothing": 3}),
    "V3 missing 0.2": ("v3", {"missing_weight": 0.2, "smoothing": 3}),
    "V3 missing 0.4": ("v3", {"missing_weight": 0.4, "smoothing": 3}),
    "V3 missing 0.6": ("v3", {"missing_weight": 0.6, "smoothing": 3}),
    "V3 smoothing 2": ("v3", {"missing_weight": 0.4, "smoothing": 2}),
    "V3 smoothing 5": ("v3", {"missing_weight": 0.4, "smoothing": 5})
}

# display method and parameter settings for each configuration
display(pd.DataFrame([{"configuration": name, "method": method, **settings} for name, (method, settings) in configurations.items()]).fillna("-"))

,configuration,method,coverage_weight,missing_weight,common_weight,missing_method,smoothing
0,Baseline,baseline,-,-,-,-,-
1,Combined V1,v1v2,0.3,0.2,0.1,v1,-
2,Combined V2,v1v2,0.5,0.4,0.1,-,-
3,Coverage only,v1v2,0.5,0.0,0.0,-,-
4,V3 missing 0.0,v3,-,0.0,-,-,3.0
5,V3 missing 0.2,v3,-,0.2,-,-,3.0
6,V3 missing 0.4,v3,-,0.4,-,-,3.0
7,V3 missing 0.6,v3,-,0.6,-,-,3.0
8,V3 smoothing 2,v3,-,0.4,-,-,2.0
9,V3 smoothing 5,v3,-,0.4,-,-,5.0


In [11]:
%%time
# EVALUATE RANDOM AND MANUAL QUERIES


#store results for each model, map each model to name
# store results for each model, map each model to name
evaluation_results = []
models = {"TF-IDF": (tfidf_model, "tfidf_similarity"), "Jaccard": (jaccard_model, "jaccard_similarity"), "MiniLM": (minilm_model, "minilm_similarity")}

# use the same evaluation for random and manual queries
query_sets = {"random": validation_query_set, "manual": manual_query_set}

# run random and manual queries through same retrieval models
for experiment, query_set in query_sets.items():
    # go through every query
    for query_number, query_row in query_set.iterrows():
        user_input = query_row["query"]
        # calculate relevance across dataset once for this query
        relevance_scores, binary_relevance, total_relevant = prepare_query_relevance(user_input)

        # go through every retrieval model
        for model_name, (model_function, similarity_column) in models.items():
            # retrieve top 100 recipes using original model similarity
            baseline_candidates = model_function(user_input, candidate_k)
            # calculate features once per model and query - reuse for every configuration
            features = prepare_reranking_features(baseline_candidates, user_input, similarity_column)

            # evaluate each configuration using the same 100 baseline candidates
            for configuration_name, (method, settings) in configurations.items():
                # keep baseline ranking or apply selected reranking method and settings
                recommendations = baseline_candidates if method == "baseline" else rerank_configuration(features, **settings) if method == "v1v2" else rerank_v3(features, **settings)

                # calculate metrics separately for top 5 and top 10
                for k in evaluation_k:
                    metrics = evaluate_recommendations(recommendations, user_input, k, relevance_scores, binary_relevance, total_relevant)
                    evaluation_results.append({"experiment": experiment, "seed": validation_seed if experiment == "random" else None, "query_id": query_row["query_id"], "group": query_row.get("group", "random"), "query_length": query_row["query_length"], "query": user_input, "model": model_name, "configuration": configuration_name, "k": k, "total_relevant": total_relevant, **metrics})

    # display progress after completing each query set
    print(f"completed {experiment} queries: {len(query_set)}")

# convert results to df
evaluation_results = pd.DataFrame(evaluation_results)



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11604.84it/s]


completed random queries: 200
completed manual queries: 20
CPU times: total: 24min 40s
Wall time: 3min 12s


In [12]:
# SUMMARISE VALIDATION EVALUATION

# keep random and manual results separate
# random query results
random_results = evaluation_results[evaluation_results["experiment"] == "random"].copy()
# manual query results
manual_results = evaluation_results[evaluation_results["experiment"] == "manual"].copy()

# mean and standard deviation across the 50 source queries for each length
random_summary = random_results.groupby(["model", "configuration", "query_length", "k"])[["precision", "recall", "ndcg", "coverage", "missing"]].agg(["mean", "std"]).round(4)
# flatten mean and std column names for display and csv
random_summary.columns = ["_".join(column) for column in random_summary.columns]
# turn grouped model and query settings back into columns
random_summary = random_summary.reset_index()

# results for each fixed manual group
manual_summary = manual_results.groupby(["group", "model", "configuration", "query_length", "k"])[["precision", "recall", "ndcg", "coverage", "missing"]].mean().round(4).reset_index()

# keep the original validation summary variable for comparison
# retain the original combined results and summary
evaluation_summary = random_summary

display(random_summary)
display(manual_summary)


,model,configuration,query_length,k,precision_mean,precision_std,recall_mean,recall_std,ndcg_mean,ndcg_std,coverage_mean,coverage_std,missing_mean,missing_std
0,Jaccard,Baseline,3,5,0.304,0.2563,0.4212,0.4256,0.7383,0.2107,0.7160,0.1339,2.748,1.0735
1,Jaccard,Baseline,3,10,0.238,0.2355,0.5125,0.4273,0.7778,0.1839,0.6913,0.1386,3.126,1.1482
2,Jaccard,Baseline,4,5,0.292,0.2329,0.7012,0.4013,0.8633,0.1824,0.7240,0.1341,3.284,1.3335
3,Jaccard,Baseline,4,10,0.198,0.2025,0.7588,0.3645,0.8822,0.1537,0.6750,0.1492,3.480,1.1636
4,Jaccard,Baseline,5,5,0.260,0.1471,0.8900,0.2877,0.9495,0.1234,0.7024,0.1360,3.412,1.3064
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235,TF-IDF,V3 smoothing 5,4,10,0.306,0.3334,0.8869,0.2851,0.9892,0.0312,0.7555,0.1680,5.596,1.7320
236,TF-IDF,V3 smoothing 5,5,5,0.316,0.2558,0.9250,0.2449,0.9983,0.0124,0.7720,0.1433,5.464,1.8798
237,TF-IDF,V3 smoothing 5,5,10,0.198,0.2511,0.9340,0.2187,0.9930,0.0316,0.7064,0.1625,5.696,1.7378
238,TF-IDF,V3 smoothing 5,6,5,0.272,0.2051,0.9617,0.1896,1.0000,0.0000,0.7353,0.1366,5.212,2.0180


,group,model,configuration,query_length,k,precision,recall,ndcg,coverage,missing
0,beef_rice,Jaccard,Baseline,3,5,0.4,0.1053,0.5182,0.8000,3.0
1,beef_rice,Jaccard,Baseline,3,10,0.3,0.1579,0.4957,0.7667,3.3
2,beef_rice,Jaccard,Baseline,4,5,0.4,0.1250,0.5182,0.8000,2.8
3,beef_rice,Jaccard,Baseline,4,10,0.3,0.1875,0.4957,0.8000,3.2
4,beef_rice,Jaccard,Baseline,5,5,0.6,0.3000,0.7443,0.8800,3.6
...,...,...,...,...,...,...,...,...,...,...
1195,white_cake,TF-IDF,V3 smoothing 5,4,10,1.0,0.0277,0.8805,1.0000,2.3
1196,white_cake,TF-IDF,V3 smoothing 5,5,5,1.0,0.0400,1.0000,1.0000,1.6
1197,white_cake,TF-IDF,V3 smoothing 5,5,10,1.0,0.0800,0.8413,1.0000,2.2
1198,white_cake,TF-IDF,V3 smoothing 5,6,5,1.0,0.0833,0.8416,1.0000,1.8


In [13]:
# SAVE VALIDATION EVALUATION RESULTS

# save random and manual results separately
random_results.to_csv("evaluation3_randomquery.csv", index=False)
random_summary.to_csv("evaluation3_randomquery_summary.csv", index=False)
manual_results.to_csv("evaluation3_manual_userqueries.csv", index=False)
manual_summary.to_csv("evaluation3_manual_userqueries_summary.csv", index=False)
